In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import solve_ivp
from scipy import stats
from scipy.optimize import curve_fit

Import the data:

In [6]:
data = {
    "Romulus_N2": pd.read_csv("../Data/Romulus_n2.csv"),
    "Romulus_O2": pd.read_csv("../Data/Romulus_o2.csv"),
    "Remus_N2": pd.read_csv("../Data/Remus_n2.csv"),
    "Remus_O2": pd.read_csv("../Data/Remus_o2.csv")
}


The linearised Langmuir equation is $\frac{p}{q}$ = $\frac{1}{{q_s}k}$ + $\frac{1}{q_s}p$, and the linearised Freundlich equation is $ln$ $q$ = $ln$ $k$ + $\frac{1}{m}$ $ln$ $p$.


We must first remove the row containing zeros. Then we transform the data into y and x.

In [9]:
P_raw = {}
q_raw = {}

x_langmuir = {}
y_langmuir = {}

x_freundlich = {}
y_freundlich = {}

for name, df in data.items():
    df = df[df["P [Pa]"] > 0]
    P = df["P [Pa]"].to_numpy()
    q = df["q [mol kg-1]"].to_numpy()
    P_raw[name] = P
    q_raw[name] = q
    x_langmuir[name] = P
    y_langmuir[name] = P/q
    x_freundlich[name] = np.log(P)
    y_freundlich[name] = np.log(q)

In [10]:
def langmuir_equation(P,qs,K):
    return qs*K*P / (1+K*P)
def freundlich_equation(P,m,K):
    return K*P**(1/m)

Fit the pure-component data to the linearised versions of the Langmuir and Freundlich isotherms and provide the functional form along with the estimated parameters (with the correct units). 

In [12]:
rows = []
for name in x_langmuir:
    results_langmuir = stats.linregress(x_langmuir[name],y_langmuir[name])
    results_freundlich = stats.linregress(x_freundlich[name],y_freundlich[name])

    qs_langmuir = 1 / results_langmuir.slope
    K_langmuir = 1 / (qs_langmuir * results_langmuir.intercept)

    m_freundlich = 1 / results_freundlich.slope
    K_freundlich = np.exp(results_freundlich.intercept)
    
    rows.append({
        "data_set": name,
        "slope_langmuir": results_langmuir.slope,
        "intercept_langmuir": results_langmuir.intercept,
        "R2_langmuir": results_langmuir.rvalue**2,
        "qs_langmuir": qs_langmuir,
        "K_langmuir": K_langmuir,
        "slope_freundlich": results_freundlich.slope,
        "intercept_freundlich": results_freundlich.intercept,
        "R2_freundlich": results_freundlich.rvalue**2,
        "m_freundlich": m_freundlich,
        "K_freundlich": K_freundlich,
    })

results = pd.DataFrame(rows)
pd.set_option("display.float_format", "{:.4e}".format)
display(results)

,data_set,slope_langmuir,intercept_langmuir,R2_langmuir,qs_langmuir,K_langmuir,slope_freundlich,intercept_freundlich,R2_freundlich,m_freundlich,K_freundlich
0,Romulus_N2,4.1649e-01,8.0222e+04,9.9654e-01,2.4010e+00,5.1917e-06,7.5151e-01,-9.1556e+00,9.8836e-01,1.3307e+00,1.0563e-04
1,Romulus_O2,3.3078e-01,5.6646e+05,9.2975e-01,3.0232e+00,5.8394e-07,9.2793e-01,-1.2550e+01,9.9989e-01,1.0777e+00,3.5452e-06
2,Remus_N2,3.5929e-01,3.5576e+05,9.9879e-01,2.7833e+00,1.0099e-06,7.7541e-01,-1.0278e+01,9.9509e-01,1.2896e+00,3.4365e-05
3,Remus_O2,2.4199e-01,9.1636e+05,9.2127e-01,4.1325e+00,2.6407e-07,9.2688e-01,-1.2918e+01,9.9988e-01,1.0789e+00,2.4524e-06


calculate the standard error propogation of the slope and intercept errors ont the parameter voalues attained. Langmuir, from $q_s = 1/s$ and $K = s/c$:

$$\sigma_{q_s} = \frac{\sigma_s}{s^2}$$

$$\sigma_K = K\sqrt{\left(\frac{\sigma_s}{s}\right)^2 + \left(\frac{\sigma_c}{c}\right)^2 - \frac{2\,\mathrm{cov}(s,c)}{s\,c}}$$

Freundlich, from $m = 1/s$ and $K = e^{c}$:

$$\sigma_m = \frac{\sigma_s}{s^2}$$

$$\sigma_K = K\,\sigma_c$$

where $s$ is the slope, $c$ is the intercept, and $\sigma = \sqrt{\mathrm{variance}}$.

Then append these standard error values to the table above.

**Component mass balance**
$$\frac{\partial y_i}{\partial t} = D_L \frac{\partial^2 y_i}{\partial z^2} - \frac{\partial (y_i v)}{\partial z} - \frac{R T \rho_s (1-\varepsilon)}{P\,\varepsilon}\frac{\partial q_i}{\partial t}, \qquad \sum_i y_i = 1 \tag{1}$$

**Axial dispersion**
$$D_L = 0.7 D_m + v_{in} r_p \tag{2}$$

**Boundary conditions**
$$z = 0: \quad y_i(z=0) = y_{i,feed} + \frac{D_L}{v}\left.\frac{\partial y_i}{\partial z}\right|_{z=0} \tag{3}$$

$$z = L: \quad \left.\frac{\partial y_i}{\partial z}\right|_{z=L} = 0 \tag{4}$$

**Solid phase mass balance (LDF)**
$$\frac{\partial q_i}{\partial t} = k_i\left(q_i^{*} - q_i\right) \tag{5}$$